In [1]:
import os
import pandas as pd

BASE_DIR = os.path.abspath("..")

file_path = os.path.join(
    BASE_DIR,
    "data",
    "intermediate",
    "sih_raw_concat.parquet"
)

COLUMNS = ['UF_ZI', 'COD_IDADE', 'IDADE', 'SEXO', 'RACA_COR',
           'CAR_INT', 'MORTE', 'DIAG_PRINC', 'DIAGSEC1', 'DIAGSEC2',
           'DIAGSEC3', 'DIAGSEC4', 'DIAGSEC5', 'DIAGSEC6', 
           'DIAGSEC7', 'DIAGSEC8', 'DIAGSEC9', 'CNES', 'ESPEC']

DIAG_COLS = ['DIAG_PRINC', 'DIAGSEC1', 'DIAGSEC2', 'DIAGSEC3', 'DIAGSEC4',
              'DIAGSEC5', 'DIAGSEC6', 'DIAGSEC7', 'DIAGSEC8', 'DIAGSEC9']

pd.set_option('display.max_columns', None)

In [ ]:
def load_SIH(directory, columns=None):
    if not os.path.isdir(directory):
        print("> Diretorio de entrada nao foi encontrado.")
        return None

    dataframes = []

    print("> Lendo arquivos parquet...")
    for root, dirs, files in os.walk(directory):

        for f in files:
            if f.endswith(".parquet"):
                path = os.path.join(root, f)
                try:
                    dataframe = pd.read_parquet(path, engine="pyarrow", columns=columns)

                    dataframes.append(dataframe)
                except Exception as e:
                    print(f">> Falha ao ler arquivo: {path}")
                    print(f">> Exception: {e}")
                    continue
    print(f"> {len(dataframes)} arquivos lidos.")

    if dataframes:
        print("> Concatenando arquivos parquet...")
        return pd.concat(dataframes, ignore_index=True)
    else:
        print("> Nenhum arquivo parquet encontrado.")
        return None

In [ ]:
raw = pd.read_parquet(file_path, engine="pyarrow")

In [ ]:
raw

In [ ]:
total_original = raw.index.size
prop_original = raw['MORTE'].value_counts(normalize=True) * 100
print(f"> Quantidade total de linhas: {total_original}")
print("> Proporcao de obito/alta: ")
print(prop_original)
print()

dedup = raw.drop_duplicates()

total_dedup = dedup.index.size
prop_dedup = dedup['MORTE'].value_counts(normalize=True) * 100
print(f"> Quantidade total de linhas apos deduplicacao: {total_dedup}")
print("> Proporcao de obito/alta: ")
print(prop_dedup)

In [ ]:
for col in raw.columns:
    nulls = raw[col].isna().sum()
    empty = raw[col].astype(str).str.strip().eq("").sum()
    datatype = raw[col].dtype

    print(f"> Coluna {col}:")
    print(f"    > Tipo: {datatype}")
    print(f"    > Nulos (NaN): {nulls}")
    print(f"    > Vazios (''): {empty}")
    print(f"    > Total ausentes: {nulls + empty}")
    print()

In [ ]:
print(raw.value_counts("UF_ZI"))
print(f"> Tamanho dos codigos de municipios gestores:\n{raw["UF_ZI"].str.len().value_counts()}")
print(r"> Codigos que comecam com 0: ")
print(raw.loc[raw["UF_ZI"].str.match(r"^0", na=False), "UF_ZI"])

In [ ]:
print(raw.value_counts("IDADE"))
raw["IDADE"] = pd.to_numeric(raw["IDADE"],errors="coerce").astype("int64")
print(f"> Idades nulas apos conversao para int: {raw["IDADE"].isna().sum()}")

In [12]:
print(raw.value_counts("COD_IDADE"))

COD_IDADE
4    3151592
2     109495
3      72604
5       1761
Name: count, dtype: int64


In [6]:
print(raw.value_counts("SEXO"))

SEXO
3    1856500
1    1478952
Name: count, dtype: int64


In [7]:
print(raw.value_counts("RACA_COR"))

RACA_COR
01    1683985
03    1378527
02     195312
99      41858
04      34881
05        889
Name: count, dtype: int64


In [8]:
print(raw.value_counts("CAR_INT"))

CAR_INT
02    2370965
01     927400
06      25465
05      11598
03         18
04          6
Name: count, dtype: int64


In [9]:
print(raw.value_counts("MORTE"))

MORTE
0    3175385
1     160067
Name: count, dtype: int64


In [ ]:
for col in DIAG_COLS:
    print(f"> Tamanho dos codigos CID-10 em {col}:\n{raw[col].str.len().value_counts()}")
    print(r"> Valores que nao correspondem ao formato ^[A-Z]\d{2,3}$: ")
    print(raw.loc[~raw[col].str.match(r"^[A-Z]\d{2,3}$", na=False), col])
    print("> Retirando espaços vazios...")
    raw[col] = raw[col].str.replace(r"\s+", "", regex=True)
    print(r"> Valores que nao correspondem ao formato ^[A-Z]\d{2,3}$: ")
    print(raw.loc[~raw[col].str.match(r"^[A-Z]\d{2,3}$", na=False), col])
    print()

> Tamanho dos codigos CID-10 em DIAG_PRINC:
DIAG_PRINC
4    3335452
Name: count, dtype: int64
> Valores que não correspondem ao formato ^[A-Z]\d{2,3}$: 
16         T07 
20         N61 
21         C20 
49         A09 
83         R51 
           ... 
3335433    A87 
3335434    J20 
3335437    J21 
3335439    J90 
3335441    B99 
Name: DIAG_PRINC, Length: 368797, dtype: str
> Retirando espaços vazios...
> Valores que não correspondem ao formato ^[A-Z]\d{2,3}$: 
Series([], Name: DIAG_PRINC, dtype: str)

> Tamanho dos codigos CID-10 em DIAGSEC1:
DIAGSEC1
4    3335452
Name: count, dtype: int64
> Valores que não correspondem ao formato ^[A-Z]\d{2,3}$: 
0              
1              
2              
3              
4              
           ... 
3335444        
3335445        
3335447        
3335449        
3335450        
Name: DIAGSEC1, Length: 2198645, dtype: str
> Retirando espaços vazios...
> Valores que não correspondem ao formato ^[A-Z]\d{2,3}$: 
0           
1           
2          

In [3]:
print(raw.value_counts("CNES"))
print(f"> Tamanho dos codigos CNES:\n{raw["CNES"].str.len().value_counts()}")

CNES
2078015    48089
2077396    45194
2082187    40872
2688689    28320
2079798    27125
           ...  
6630537        3
7965192        3
2519089        2
2080699        1
2079038        1
Name: count, Length: 934, dtype: int64
> Tamanho dos codigos CNES:
CNES
7    3335452
Name: count, dtype: int64


In [4]:
print(raw.value_counts("ESPEC"))

ESPEC
03    1150805
01    1131022
02     449921
07     321984
09     142552
05      77979
04      37473
12       5905
87       5269
08       4121
06       3791
14       2608
10       1188
13        831
11          3
Name: count, dtype: int64
